# Nothing Lost in Translation

This file is meant to serve as a focused discussion of the implementation journey of the project covering the path I took, the results I achieved and how some of the decisions were pitfalls that may be helpful to know about if anyone is reading this in detail plans to do something similar. 

Author: Anupreet Singh  
Base project: GA1 implementation for NLP 673

## Section 0 - Setting up the environment

For running on a local machine, use a Python virtual environment so the dependencies stay isolated:

```bash
cd <path-to-this-project>
python3 -m venv .venv
source .venv/bin/activate
```

Then select `.venv` as this notebook's kernel.

In [22]:
## Installing required libraries
# numpy>=2 now works with torch (back in 2024 it didn't, and the original project had to pin numpy<2).
# ipywidgets and tqdm are only needed for progress bars in a local Jupyter session.
%pip install transformers "numpy>=2" torch "accelerate>=0.26.0" sacrebleu ipywidgets sentencepiece tqdm datasets

In [ ]:
## Importing libraries
from transformers import MarianMTModel, MarianTokenizer # Classes that help us load Marian MT models and tokenizers
from transformers.modeling_outputs import BaseModelOutput  # lets us hand pre-computed encoder states to the decoder in grid beam search
import torch
import torch.nn.functional as F
from datasets import load_dataset
from tqdm import tqdm
import sacrebleu

## Section 1 — The Dataset

The dataset is WMT16 EN→RU: ~1.5M training pairs and 2,998 validation pairs. Its layout is worth one note because every later cell indexes into it: the data lives in a single `translation` column whose rows are dictionaries with one entry per language, so `dataset["train"]["translation"][i]["en"]` walks split → column → row → language. (Printing a split shows a structural summary, not the rows themselves.)

In [24]:
## Downloading the dataset
dataset = load_dataset("wmt/wmt16", "ru-en")  # ~1.5M train pairs, 2,998 validation pairs

# Bigger alternative:
# dataset = load_dataset("wmt/wmt19", "ru-en")

# Seeing the structure of the dataset
print(dataset["validation"])
for i in range(5):
    print(dataset["train"]["translation"][i]["en"])

Dataset({
    features: ['translation'],
    num_rows: 2818
})
iron cement is a ready for use paste which is laid as a fillet by putty knife or finger in the mould edges (corners) of the steel ingot mould.
iron cement protects the ingot against the hot, abrasive steel casting process.
iron cement is freshly applied after each steel pour in a coating thickness of approx. ~ 2-3 mm.
a fire restant repair cement for fire places, ovens, open fireplaces etc.
Translator Internet is a Toolbar for MS Internet Explorer.


## Section 2: Landing on a Model 

### Why I dropped mT5 as a Candidate Model

The first modeling attempt was `google/mt5-base`. 

mT5 is pretrained on multilingual text, but it is not a translation model out of the box. There are two ways to turn it into one:

- Prefix the input with a task description and rely on prompting.
- Fine-tune it on the 1.5 million English-Russian training pairs so it learns the translation task directly.

I tried the practical route and fine-tuned mT5. The result did not hold up: after hours of CPU training, the fine-tuned model still trailed a purpose-built EN->RU translation model. That result redirected the project away from mT5.

### Fine-tuning mT5 (kept for the record)

The fine-tuning pipeline: tokenize both sides of every pair to a fixed length of 128 — `padding="max_length"` because `Trainer` requires uniform sequence lengths within a batch, and truncation at 128 costs little when the goal is *comparing* models rather than maximizing quality — attach the target ids as `labels`, and hand everything to Hugging Face's `Trainer`. `Trainer` uses the AdamW optimizer by default; `weight_decay` penalizes large weights so no single feature can dominate predictions, a standard guard against overfitting.

The code stays here behind a flag as part of the research record; nothing after this cell depends on it.

In [25]:
## Fine-tuning mT5 on WMT16 (historical step)
RUN_MT5_FINETUNE = False  # takes hours; the final pipeline does not use the result. Flip to True to reproduce.

if RUN_MT5_FINETUNE:
    from transformers import MT5ForConditionalGeneration, MT5Tokenizer, Trainer, TrainingArguments

    mt5_tokenizer = MT5Tokenizer.from_pretrained("google/mt5-base")
    mt5_model = MT5ForConditionalGeneration.from_pretrained("google/mt5-base")

    def preprocess_function(examples):
        source = mt5_tokenizer([ex["en"] for ex in examples["translation"]],
                               padding="max_length", truncation=True, max_length=128)
        target = mt5_tokenizer([ex["ru"] for ex in examples["translation"]],
                               padding="max_length", truncation=True, max_length=128)
        source["labels"] = target["input_ids"]  # the target ids are what the Trainer optimizes against
        return source

    # .map processes in batches of 1000 and caches intermediate results on disk
    train_T = dataset["train"].map(preprocess_function, batched=True).remove_columns(["translation"])
    eval_T = dataset["validation"].map(preprocess_function, batched=True).remove_columns(["translation"])

    training_args = TrainingArguments(
        output_dir="artifacts/mt5-checkpoints",   # checkpoints = model state saved every save_steps
        save_steps=10_000,
        save_total_limit=2,
        logging_steps=500,
        eval_strategy="epoch",                    # named evaluation_strategy on transformers < 4.41
        learning_rate=5e-5,
        per_device_train_batch_size=10,
        per_device_eval_batch_size=4,
        num_train_epochs=3,
        weight_decay=0.01,
    )
    trainer = Trainer(
        model=mt5_model,
        args=training_args,
        train_dataset=train_T,
        eval_dataset=eval_T.select(range(10)),
        tokenizer=mt5_tokenizer,
    )
    trainer.train()

    mt5_model.save_pretrained("artifacts/mt5-finetuned")
    mt5_tokenizer.save_pretrained("artifacts/mt5-finetuned")

### The switch to MarianMT

The redirect was cheap because of what this research is actually about. Every question from here on concerns *decoding* — how the output sequence is searched for, and how hard constraints can be injected into that search — and none of it depends on which seq2seq model produces the logits. 

`MarianMTModel` and `MarianTokenizer` are classes from `transformers` package that help us load components such as models tokenizer from models that are packaged using the Marian Architecture. `Helsinki-NLP/opus-mt-en-ru` is one such Marian Model. `Helsinki-NLP/opus-mt-en-ru` is small, purpose-built for this language pair, and strong with no training at all, so it provides a better experimental foundation for free. Everything below uses it.

### An Important MarianTokenizer detail

`MarianTokenizer` bundles two SentencePiece models: `source.spm` (English) and `target.spm` (Russian). A bare `tokenizer(...)` call always segments with the **source** one. Segment Russian text that way and you get character shrapnel or `<unk>` pieces — token sequences the decoder never produces.

This detail broke the project's first constrained-decoding implementation: references, mined phrases, and hypotheses were all encoded with the English segmenter, so every constraint pushed into the decoder was expressed in a vocabulary it never emits. Constrained decodes were poisoned, and because the forced phrase never actually appeared in the output, the same phrase was re-mined cycle after cycle — BLEU *fell* when constraints were applied. (The full post-mortem is in the development notebook, Section 6.)

The rule that fixes it: **every target-side string goes through `encode_target()`**. The sanity check defined next prints both segmentations side by side, so the difference is never invisible again.

## Section 3 — A trustworthy baseline: `.generate()` and BLEU

Before replacing the decoder, we need the number to beat. This section loads MarianMT, prepares the evaluation subset — the first `N_EVAL` sentences of the validation split, the knob that trades runtime against BLEU stability — translates it with the library's own beam search, and scores it. Every later experiment is read as a delta from this score.

In [26]:
## Loading the Helsinki tokenizer and model
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ru")
model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-ru")

special_ids = set(tokenizer.all_special_ids)  # used throughout to strip pad/eos/unk from id sequences

## Moving the model to the correct device
# Auto-detect: CUDA (Linux/Colab GPU) -> MPS (Apple Silicon) -> CPU fallback
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")
model.to(device)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Using device: cuda


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(62518, 512, padding_idx=62517)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(62518, 512, padding_idx=62517)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [27]:
def encode_target(text, **kwargs):
    """Tokenize TARGET-language (Russian) text with the target spm.

    A bare tokenizer(...) call uses the *source* (English) SentencePiece
    model, which is wrong for references, mined constraint phrases, and
    hypotheses. Newer transformers versions take text_target=; older ones
    need the as_target_tokenizer() context manager.
    """
    try:
        return tokenizer(text_target=text, **kwargs)
    except TypeError:  # transformers < 4.22
        with tokenizer.as_target_tokenizer():
            return tokenizer(text, **kwargs)


def sanity_check_target_tokenizer(sample):
    """Print how the source vs. target spm segment a Russian sentence.

    With the source spm you should see character shrapnel or <unk> pieces;
    with the target spm, normal Russian subwords. This is the difference
    that broke the original run.
    """
    src_pieces = tokenizer.convert_ids_to_tokens(
        tokenizer(sample, add_special_tokens=False)["input_ids"]
    )
    tgt_pieces = tokenizer.convert_ids_to_tokens(
        encode_target(sample, add_special_tokens=False)["input_ids"]
    )
    print("  source-spm segmentation:", src_pieces)
    print("  target-spm segmentation:", tgt_pieces)
    if tokenizer.unk_token in tgt_pieces:
        print("  WARNING: <unk> even in the target segmentation")

### Run Configuration

In [40]:
## Run configuration
N_EVAL = 100       # how many validation sentences to run on; raise for more stable BLEU, lower for speed
CYCLES = 4          # pick-revise cycles (one new constraint mined per sentence per cycle)
NUM_BEAMS = 4       # Hokamp & Liu used 10; raise this for a closer (slower) reproduction
MAX_LENGTH = 128
MINE_MODE = "word"  # "word" or "token" -- the granularity at which missing phrases are mined

model.eval()  # disables dropout so the whole network is used, deterministically, at inference

MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(62518, 512, padding_idx=62517)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(62518, 512, padding_idx=62517)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [41]:
## Preparing the evaluation subset
_eval = dataset["validation"]
src_texts = [row["en"] for row in _eval["translation"]][:N_EVAL]
tgt_texts = [row["ru"] for row in _eval["translation"]][:N_EVAL]
refs_bleu = [tgt_texts]  # sacrebleu expects a list of reference *sets*, one set per possible reference translation

_src_tok = tokenizer(
    src_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
).to(device)
src_ids = _src_tok["input_ids"]
src_mask = _src_tok["attention_mask"]

# References are Russian -> they must go through the target spm
ref_ids = encode_target(
    tgt_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
).to(device)["input_ids"]

print("Tokenizer sanity check (source vs. target spm on the first Russian reference):")
sanity_check_target_tokenizer(tgt_texts[0])

Tokenizer sanity check (source vs. target spm on the first Russian reference):
  source-spm segmentation: ['▁', 'П', 'р', 'е', 'м', 'ь', 'е', 'р', '-', 'м', 'и', 'н', 'и', 'с', 'т', 'р', 'ы', '▁', 'И', 'н', 'д', 'и', 'и', '▁', 'и', '▁', 'Я', 'п', 'о', 'н', 'и', 'и', '▁', 'в', 'с', 'т', 'р', 'е', 'ч', 'а', 'ю', 'т', 'с', 'я', '▁', 'в', '▁', 'Т', 'о', 'к', 'и', 'о']
  target-spm segmentation: ['▁Премьер', '-', 'министр', 'ы', '▁Индии', '▁и', '▁Японии', '▁встречаются', '▁в', '▁Токио']


Two scoring conventions to know before the baseline run. `sacrebleu.corpus_bleu(hypotheses, references)` takes the hypotheses as a flat list of strings and the references as a list of reference *sets* (a list of lists), so each hypothesis can be judged against multiple acceptable translations. And a trap learned the hard way: sacrebleu does **not** raise an error when the hypothesis and reference counts mismatch — check the lengths yourself before trusting any score.

In [42]:
## Translating the subset with the library's beam search
generation_batch_size = 30  # tune to the memory available on your device

generate_outputs = []
for i in tqdm(range(0, N_EVAL, generation_batch_size), desc=".generate baseline"):
    out = model.generate(
        input_ids=src_ids[i:i + generation_batch_size],
        attention_mask=src_mask[i:i + generation_batch_size],
        num_beams=NUM_BEAMS,
        max_length=MAX_LENGTH,
        early_stopping=True,
    )
    generate_outputs.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

generate_bleu = sacrebleu.corpus_bleu(generate_outputs, refs_bleu).score
print(f"BLEU (.generate, first {N_EVAL} validation sentences): {generate_bleu:.2f}")

.generate baseline: 100%|██████████| 4/4 [00:05<00:00,  1.41s/it]

BLEU (.generate, first 100 validation sentences): 24.23


## Section 4 — Beam search from scratch

`model.generate()` cannot be reached into; forcing a phrase into the output requires owning the decoding loop. The loop below drives the model's raw forward pass token by token: the source ids enter the encoder once → at each step the decoder receives the tokens generated so far and returns logits for the next position → `log_softmax` turns those into log-probabilities that can be added across steps → every live hypothesis proposes its top-`num_beams` continuations → candidates from the same source sentence compete, and the best `num_beams` survive with updated running scores → the loop ends at EOS or `max_length` → the best beam is decoded back into text.

Anatomy useful for reading the code: for the Helsinki model `eos_token_id=0`, `unk=1`, `pad=62517`, and `decoder_start_token_id=62517` — the pad id doubles as the decoder's start token. One decoder invocation that produces next-token logits is a *forward pass*.

Two design points:

- `scratch_beam_search` is deliberately unconstrained — it exists to reproduce `.generate()` and, in doing so, to open the decoding loop up for surgery. Both constrained decoders later in the notebook start from it: stamping (Section 6) grafts a constraint queue onto this exact loop, and Grid Beam Search (Section 7) rebuilds the search around a coverage grid while keeping the same per-step anatomy.
- Finished beams are frozen (extended with pad at zero cost) rather than being allowed to generate past EOS.

The point of this section is trust: if the hand-written loop matches `.generate()` on BLEU — the two scores are printed together below — it is a faithful re-implementation and safe to build on.

In [43]:
def scratch_beam_search(
    input_ids,
    attention_mask,
    num_beams=NUM_BEAMS,
    max_length=MAX_LENGTH,
    early_stopping=True,
):
    """Vanilla beam search from scratch: one forward pass per step,
    log-probs summed along each hypothesis, and the top-`num_beams`
    candidates per source sentence surviving each step.

    This is the trunk decoder of the project. It has to match
    `.generate()` on BLEU (checked below) before anything is built on
    it; the constrained decoders in Sections 6 and 7 both grow out of
    this loop. Finished beams are frozen (extended with pad at zero
    cost) so nothing is generated after EOS.
    """
    batch = input_ids.shape[0]
    eos_id = tokenizer.eos_token_id
    pad_id = model.config.pad_token_id

    input_ids = input_ids.repeat_interleave(num_beams, dim=0)
    attention_mask = attention_mask.repeat_interleave(num_beams, dim=0)

    decoder_input_ids = torch.full(
        (batch * num_beams, 1),
        model.config.decoder_start_token_id,
        dtype=torch.long,
        device=device,
    )
    beam_scores = torch.zeros(batch * num_beams, device=device)
    finished = torch.zeros(batch * num_beams, dtype=torch.bool, device=device)

    start = True

    for step in range(max_length):
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids,
            )
            log_probs = F.log_softmax(outputs.logits[:, -1, :], dim=-1)

        gen_lp = log_probs
        gen_lp[:, pad_id] = float("-inf")
        if finished.any():
            # freeze finished beams: extend with pad at zero cost
            gen_lp[finished] = float("-inf")
            gen_lp[finished, pad_id] = 0.0

        top_lp, top_ids = torch.topk(gen_lp, num_beams, dim=1)

        if start:
            # diversify beams with the top-k first tokens of beam 0
            new_tokens = top_ids[::num_beams].reshape(-1, 1)
            decoder_input_ids = torch.cat([decoder_input_ids, new_tokens], dim=1)
            beam_scores = beam_scores + top_lp[::num_beams].reshape(-1)
            finished = new_tokens.squeeze(1) == eos_id
            start = False
        else:
            cand = (top_lp + beam_scores.unsqueeze(1)).view(
                batch, num_beams * num_beams
            )
            new_scores, flat_idx = torch.topk(cand, num_beams, dim=1)

            beam_idx = flat_idx // num_beams
            base = (
                torch.arange(batch, device=device).unsqueeze(1) * num_beams
            )
            src = (beam_idx + base).view(-1)

            tok_mat = top_ids.view(batch, num_beams * num_beams)
            new_tok = tok_mat[
                torch.arange(batch, device=device).unsqueeze(1), flat_idx
            ].view(batch * num_beams, 1)

            decoder_input_ids = torch.cat([decoder_input_ids[src], new_tok], dim=1)
            beam_scores = new_scores.view(-1)
            finished = finished[src] | (new_tok.squeeze(1) == eos_id)

        fin = finished.view(batch, num_beams)
        if fin.all():
            break
        if early_stopping and fin[:, 0].all():
            break

    return decoder_input_ids.view(batch, num_beams, -1)[:, 0, :]

In [44]:
## Translating the subset with the scratch decoder
base_outputs = []
for i in tqdm(range(N_EVAL), desc="scratch beam search"):
    ids = scratch_beam_search(src_ids[i:i + 1], src_mask[i:i + 1])
    base_outputs.append(tokenizer.batch_decode(ids, skip_special_tokens=True)[0])

base_bleu = sacrebleu.corpus_bleu(base_outputs, refs_bleu).score
print(f"BLEU (scratch beam search): {base_bleu:.2f}")
print(f"BLEU (.generate)          : {generate_bleu:.2f}")

scratch beam search: 100%|██████████| 100/100 [00:41<00:00,  2.40it/s]

BLEU (scratch beam search): 25.45
BLEU (.generate)          : 24.23


## Section 5 — Mining lexical constraints

With a trustworthy decoder, the question becomes: can we *guarantee* that specific reference phrases appear in the translation? 

The workflow is **pick–revise**: compare the current hypothesis against the reference → pick the longest reference n-gram (up to 3 words) missing from the hypothesis → make that phrase a hard constraint for the next decode → repeat, so each cycle adds one constraint per sentence and the translation is revised around all of them.

This section implements the *pick* half — mining. Each design rule below exists because its absence was a real failure mode in the project's first implementation (the development notebook's Section 6 tells that story in full):

- **Longest-first (3 → 2 → 1).** A longer contiguous phrase carries more information per constraint, so trigrams are preferred and shorter n-grams are fallbacks.
- **Span tracking (`used_spans`).** A reference span already used as a constraint in an earlier cycle is skipped; without this, overlapping constraints get re-mined and force duplicated text into the output.
- **Position semantics.** `position` = the number of target tokens before the phrase in the reference. The decoding loop's `step` counts generated tokens from 0, so no off-by-one correction is needed. Only stamping consumes the position — GBS places the phrase itself.
- **Two granularities.** `MINE_MODE="word"` mines on whitespace words and then tokenizes the phrase (the default: constraints align with real word boundaries); `"token"` mines directly on de-padded token ids.

In [45]:
## Small helpers shared by mining and evaluation
def strip_special(id_list):
    return [int(t) for t in id_list if int(t) not in special_ids]


def is_contiguous_subseq(needle, haystack):
    """Used after decoding to verify a constraint's ids actually appear in the output."""
    n = len(needle)
    if n == 0:
        return True
    return any(haystack[i:i + n] == needle for i in range(len(haystack) - n + 1))


def _contig(phrase, words):
    n = len(phrase)
    return any(words[i:i + n] == phrase for i in range(len(words) - n + 1))


def _overlaps(span, used_spans):
    return any(not (span[1] <= s or e <= span[0]) for s, e in used_spans)

In [46]:
def mine_word(hyp_text, ref_text, used_spans, max_words=3):
    """Longest (up to max_words) reference word n-gram missing from the
    hypothesis, skipping spans already used as constraints in earlier
    cycles (overlapping constraints would force duplicated text).

    Returns (constraint_ids, position, phrase_text, span) or Nones.
    position = number of target tokens before the phrase in the reference
    (used only by the stamping decoder; GBS places constraints itself).
    """
    hyp_words = hyp_text.split()
    ref_words = ref_text.split()

    for n in range(max_words, 0, -1):
        for i in range(len(ref_words) - n + 1):
            span = (i, i + n)
            if _overlaps(span, used_spans):
                continue

            phrase = ref_words[i:i + n]
            if _contig(phrase, hyp_words):
                continue

            # the phrase is Russian -> target spm, and drop any
            # special ids defensively
            ids = encode_target(
                " ".join(phrase),
                add_special_tokens=False,
                return_tensors="pt",
            )["input_ids"][0]
            clean = strip_special(ids.tolist())
            if not clean:
                continue

            ids = torch.tensor(clean, dtype=torch.long, device=device)

            # target spm for the prefix too, and no +1 (the stamping
            # loop's `step` counts generated tokens starting at 0)
            pos = len(
                encode_target(
                    " ".join(ref_words[:i]),
                    add_special_tokens=False,
                )["input_ids"]
            )

            return ids, pos, " ".join(phrase), span

    return None, None, None, None


def mine_token(cur_ids_row, ref_ids_row, used_spans, max_tokens=3):
    """Token-level mining (MINE_MODE='token'). Works on de-padded
    sequences and uses the same position semantics as word mode.
    Spans are in token coordinates here."""
    hyp = strip_special(cur_ids_row.tolist())
    ref = strip_special(ref_ids_row.tolist())

    def present(sub):
        n = len(sub)
        return any(hyp[i:i + n] == sub for i in range(len(hyp) - n + 1))

    for n in range(max_tokens, 0, -1):
        for i in range(len(ref) - n + 1):
            span = (i, i + n)
            if _overlaps(span, used_spans):
                continue
            sub = ref[i:i + n]
            if present(sub):
                continue
            ids = torch.tensor(sub, dtype=torch.long, device=device)
            return ids, i, tokenizer.decode(sub), span

    return None, None, None, None


def mine(hyp_text, ref_text, cur_ids_row, ref_ids_row, used_spans):
    if MINE_MODE == "word":
        return mine_word(hyp_text, ref_text, used_spans)
    return mine_token(cur_ids_row, ref_ids_row, used_spans)

## Section 6 — Fulfilling constraints I: Stamping

Mining decides *what* must appear. Fulfillment decides *where* and *how* it gets placed. 

**Stamping Beam Search** places each mined phrase(constraint) at the position it occupies in the reference translation.

`stamping_beam_search` below is Section 4's scratch decoder with a constraint schedule grafted onto the loop: constraints enter as a queue sorted by position → a constraint becomes due once decoding reaches its position (`step >= position`) and emits one token per step, so colliding positions queue up instead of overwriting each other → each forced token is still scored under the model, keeping beam scores comparable → EOS stays masked while anything is pending, so a sentence cannot end before its constraints are placed. Everything else — scoring, beam competition, frozen finished beams — is unchanged from `scratch_beam_search`, so whatever the BLEU curve does is attributable to the constraints alone.

The experiment protocol, shared with GBS in the next section: both tracks start from the same unconstrained baseline (Section 4's `base_outputs`), and each cycle mines one new constraint per sentence *from that track's own previous output*, then re-decodes with all constraints accumulated so far. Per cycle, the loop reports:

- **BLEU** — the headline metric;
- **constraints** — how many sentences still had a missing reference phrase to mine;
- **satisfied(tok)** — the fraction of constraints whose token ids appear contiguously in the output ids (did forcing actually work?);
- **in_text** — the fraction whose surface phrase survived detokenization (subword joins can alter the surface form even when the ids are present);
- **mean_len** — mean output length in tokens, a check that constraints aren't bloating the output.

The first cycle also prints before/after examples, so individual constraints can be seen landing.

In [47]:
def stamping_beam_search(
    input_ids,
    attention_mask,
    constraints,
    num_beams=NUM_BEAMS,
    max_length=MAX_LENGTH,
    early_stopping=True,
):
    """Section 4's scratch decoder extended to force-place constraint
    tokens at fixed output positions taken from the reference
    tokenization.

    Everything about scoring and beam competition is identical to
    `scratch_beam_search`; the additions (marked "stamping:" below) are
    the constraint queue, the forced-token branch, and the EOS mask
    while constraints are pending. The position comes from the
    reference -- oracle information; "Why stamping cannot leave the
    lab" below explains why that matters.
    """
    batch = input_ids.shape[0]
    eos_id = tokenizer.eos_token_id
    pad_id = model.config.pad_token_id

    input_ids = input_ids.repeat_interleave(num_beams, dim=0)
    attention_mask = attention_mask.repeat_interleave(num_beams, dim=0)

    decoder_input_ids = torch.full(
        (batch * num_beams, 1),
        model.config.decoder_start_token_id,
        dtype=torch.long,
        device=device,
    )
    beam_scores = torch.zeros(batch * num_beams, device=device)
    finished = torch.zeros(batch * num_beams, dtype=torch.bool, device=device)

    # stamping: the constraint schedule, sorted by position. It is shared
    # across the batch (this notebook decodes one sentence at a time when
    # constraints are in play).
    pending = [
        {
            "tokens": [
                int(t) for t in c["token_ids"].tolist()
                if int(t) not in special_ids
            ],
            "position": int(c["position"]),
        }
        for c in constraints
    ]
    pending = [p for p in pending if p["tokens"]]
    pending.sort(key=lambda p: p["position"])

    start = True

    for step in range(max_length):
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids,
            )
            log_probs = F.log_softmax(outputs.logits[:, -1, :], dim=-1)

        # stamping: a constraint is due once the step has reached its
        # position; emitting one token per step makes overlapping
        # constraints queue up instead of clobbering each other
        due = next((p for p in pending if step >= p["position"]), None)

        if due is not None:
            # EOS is masked whenever constraints are pending, so no beam
            # can be finished here.
            tok = due["tokens"].pop(0)
            beam_scores = beam_scores + log_probs[:, tok]  # score the forced token
            forced = torch.full(
                (batch * num_beams, 1), tok, dtype=torch.long, device=device
            )
            decoder_input_ids = torch.cat([decoder_input_ids, forced], dim=1)
            if not due["tokens"]:
                pending.remove(due)
            continue

        gen_lp = log_probs
        gen_lp[:, pad_id] = float("-inf")
        if pending:
            # stamping: can't end the sentence before every constraint is placed
            gen_lp[:, eos_id] = float("-inf")
        if finished.any():
            # freeze finished beams: extend with pad at zero cost
            gen_lp[finished] = float("-inf")
            gen_lp[finished, pad_id] = 0.0

        top_lp, top_ids = torch.topk(gen_lp, num_beams, dim=1)

        if start:
            # diversify beams with the top-k first tokens of beam 0
            new_tokens = top_ids[::num_beams].reshape(-1, 1)
            decoder_input_ids = torch.cat([decoder_input_ids, new_tokens], dim=1)
            beam_scores = beam_scores + top_lp[::num_beams].reshape(-1)
            finished = new_tokens.squeeze(1) == eos_id
            start = False
        else:
            cand = (top_lp + beam_scores.unsqueeze(1)).view(
                batch, num_beams * num_beams
            )
            new_scores, flat_idx = torch.topk(cand, num_beams, dim=1)

            beam_idx = flat_idx // num_beams
            base = (
                torch.arange(batch, device=device).unsqueeze(1) * num_beams
            )
            src = (beam_idx + base).view(-1)

            tok_mat = top_ids.view(batch, num_beams * num_beams)
            new_tok = tok_mat[
                torch.arange(batch, device=device).unsqueeze(1), flat_idx
            ].view(batch * num_beams, 1)

            decoder_input_ids = torch.cat([decoder_input_ids[src], new_tok], dim=1)
            beam_scores = new_scores.view(-1)
            finished = finished[src] | (new_tok.squeeze(1) == eos_id)

        fin = finished.view(batch, num_beams)
        if fin.all():
            break
        if early_stopping and not pending and fin[:, 0].all():
            break

    return decoder_input_ids.view(batch, num_beams, -1)[:, 0, :]

In [48]:
def run_pick_revise(decode_fn, name, base_outputs, show_examples=3):
    constraints_per = [[] for _ in range(N_EVAL)]
    spans_per = [[] for _ in range(N_EVAL)]  # reference spans already constrained
    current = list(base_outputs)

    tqdm.write(f"\n{'=' * 74}\n{name}\n{'=' * 74}")
    tqdm.write(f"base    BLEU={sacrebleu.corpus_bleu(current, refs_bleu).score:6.2f}")

    for cycle in range(CYCLES):
        if MINE_MODE == "token":
            # hypotheses are Russian -> target spm
            cur_ids = encode_target(
                current,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
            ).to(device)["input_ids"]
        else:
            cur_ids = None

        new_out = []
        n_constraints = 0
        n_sat_tok = 0
        n_sat_text = 0
        examples = []

        for i in tqdm(range(N_EVAL), desc=f"{name} cycle {cycle + 1}"):
            constraint, position, ctext, span = mine(
                current[i],
                tgt_texts[i],
                cur_ids[i] if cur_ids is not None else None,
                ref_ids[i],
                spans_per[i],
            )

            if constraint is None:
                new_out.append(current[i])
                continue

            n_constraints += 1
            spans_per[i].append(span)
            constraints_per[i].append(
                {
                    "token_ids": constraint,
                    "position": int(position),
                }
            )

            out_ids = decode_fn(
                src_ids[i:i + 1],
                src_mask[i:i + 1],
                constraints_per[i],
            )

            out_text = tokenizer.batch_decode(
                out_ids,
                skip_special_tokens=True,
            )[0]

            new_out.append(out_text)

            c_ids = strip_special(constraint.tolist())
            out_clean = strip_special(out_ids[0].tolist())
            sat_tok = is_contiguous_subseq(c_ids, out_clean)
            sat_text = ctext in out_text  # surface check: did it survive detok?
            n_sat_tok += int(sat_tok)
            n_sat_text += int(sat_text)

            if len(examples) < show_examples:
                examples.append((i, ctext, current[i], out_text, sat_tok, sat_text))

        current = new_out

        bleu = sacrebleu.corpus_bleu(current, refs_bleu).score
        mean_len = (
            sum(
                len(strip_special(
                    encode_target(o, add_special_tokens=False)["input_ids"]
                ))
                for o in current
            )
            / len(current)
        )
        rate_tok = (n_sat_tok / n_constraints * 100) if n_constraints else float("nan")
        rate_text = (n_sat_text / n_constraints * 100) if n_constraints else float("nan")

        tqdm.write(
            f"cycle {cycle + 1}  BLEU={bleu:6.2f}   "
            f"constraints={n_constraints:3d}   "
            f"satisfied(tok)={rate_tok:5.1f}%   "
            f"in_text={rate_text:5.1f}%   "
            f"mean_len={mean_len:5.1f}"
        )

        if cycle == 0:
            for idx, ctext_, before, after, st, sx in examples:
                tqdm.write(
                    f"    [{idx}] constraint={ctext_!r}  tok={st}  text={sx}"
                )
                tqdm.write(f"        before: {before}")
                tqdm.write(f"        after : {after}")

    return current

In [49]:
## Track A: pick-revise with stamping (positions copied from the reference)
stamping_outputs = run_pick_revise(stamping_beam_search, "STAMPING", base_outputs)


STAMPING
base    BLEU= 25.45


STAMPING cycle 1:   0%|          | 0/100 [00:00<?, ?it/s]

STAMPING cycle 1: 100%|██████████| 100/100 [00:40<00:00,  2.46it/s]


cycle 1  BLEU= 38.68   constraints= 99   satisfied(tok)=100.0%   in_text=100.0%   mean_len= 25.4
    [0] constraint='Премьер-министры Индии и'  tok=True  text=True
        before: Встреча премьер-министров Индии и Японии в Токио
        after : Премьер-министры Индии и Японии встречаются в Токио
    [1] constraint='Новый премьер-министр Индии,'  tok=True  text=True
        before: Новый премьер-министр Индии Нарендра Моди встречается со своим японским коллегой, Синзо Абэ, в Токио, чтобы обсудить экономические связи и связи в области безопасности в ходе своего первого крупного визита за рубеж после победы на выборах в мае.
        after : Новый премьер-министр Индии, Нарендра Моди, встречается со своим японским коллегой, Синзо Абэ, в Токио, чтобы обсудить экономические связи и связи в области безопасности в ходе своего первого крупного визита за рубеж после победы на выборах в мае.
    [2] constraint='Господин Моди находится'  tok=True  text=True
        before: Мистер Моди в пятидневно

STAMPING cycle 2: 100%|██████████| 100/100 [00:39<00:00,  2.52it/s]


cycle 2  BLEU= 56.20   constraints= 93   satisfied(tok)=100.0%   in_text=100.0%   mean_len= 25.8


STAMPING cycle 3: 100%|██████████| 100/100 [00:35<00:00,  2.82it/s]


cycle 3  BLEU= 71.95   constraints= 78   satisfied(tok)=100.0%   in_text=100.0%   mean_len= 25.3


STAMPING cycle 4: 100%|██████████| 100/100 [00:30<00:00,  3.29it/s]

cycle 4  BLEU= 81.70   constraints= 59   satisfied(tok)=100.0%   in_text=100.0%   mean_len= 25.4


### Why stamping cannot leave the lab

Stamping's per-cycle BLEU climbs dramatically — and it should. Every cycle injects another phrase taken from the reference, at the exact position the reference puts it. That second part is the flaw. The position is *oracle information*: it exists only because the reference translation is sitting in front of us to copy from. In every real use for hard lexical constraints — enforcing glossary terminology, keeping a product name intact, interactive post-editing — the user knows which phrase must appear, and never where it belongs in a sentence that hasn't been generated yet.

So stamping is a neat trick that builds on the deconstructed decoder from Section 4 and achieves very high BLEU scores, but it is not a shippable decoding method. I built it as a learning exercise before implementing real Grid Beam Search, and it earns its place here as the upper bound: the score you reach when placement comes for free.

What's needed instead is a mechanism that accepts bare phrases and lets the model discover their placement. That is exactly Grid Beam Search.

## Section 7 — Fulfilling constraints II: Grid Beam Search

GBS (Hokamp & Liu 2017) organizes the search over a grid indexed by *(timestep, coverage)*, where coverage = the number of constraint tokens emitted so far. 

The flow of one step: all live hypotheses across every coverage level are batched through the decoder in a single forward pass (the encoder ran once up front; its states are re-used via `BaseModelOutput`) → every hypothesis expands three ways, each scored by the model — **generate** (a free token, same coverage), **start** (first token of a not-yet-covered constraint, coverage + 1), **continue** (next token of the currently open constraint, coverage + 1) → the successors are re-bucketed by coverage, and only the top-`num_beams` *distinct* hypotheses per level survive to the next step.

Bookkeeping: each hypothesis is a plain dict — `ids` (tokens so far), `score` (summed log-probs), `done` (constraints fully covered), `open` (the constraint currently mid-emission, if any). Two guards keep the search honest: a hypothesis is pruned once it can no longer cover the remaining constraint tokens in the steps left, and EOS is masked until coverage is complete. The winner is the best length-normalized finished hypothesis that has covered **all** constraints.

The property that matters: constraints enter as bare phrases — no positions anywhere in the interface. This is the same information a real user could actually supply.

In [ ]:
def grid_beam_search(
    input_ids,
    attention_mask,
    constraints,
    negative_constraints=(),
    num_beams=NUM_BEAMS,
    max_length=MAX_LENGTH,
    length_penalty=1.0,
):
    """Grid Beam Search (Hokamp & Liu 2017) over a (timestep, coverage)
    grid, where coverage = constraint tokens emitted so far.

    `constraints` are bare phrases ({"token_ids": ...} dicts) that must
    appear in the output -- no positions anywhere in the interface.
    `negative_constraints` (Section 8) are phrases that must NOT appear
    as exact token sequences; they add no grid dimension, only masking,
    and every addition is marked "negative:" below.

    Each hypothesis is a plain dict: `ids` (tokens so far), `score`
    (summed log-probs), `done` (constraints fully covered), `open` (the
    constraint currently mid-emission, if any). EOS stays masked until
    coverage is complete, hypotheses that can no longer cover the
    remaining constraint tokens in the steps left are pruned, and the
    winner is the best length-normalized finished hypothesis that has
    covered all constraints.
    """
    eos_id = tokenizer.eos_token_id
    start_id = model.config.decoder_start_token_id
    pad_id = model.config.pad_token_id

    ctoks = []
    for c in constraints:
        toks = [
            int(t)
            for t in c["token_ids"].tolist()
            if int(t) not in special_ids
        ]
        if toks:
            ctoks.append(toks)

    # negative: banned token-id sequences. Matching is exact, so every
    # surface form to block (case, inflection, capitalization) needs its
    # own entry.
    btoks = []
    for c in negative_constraints:
        toks = [
            int(t)
            for t in c["token_ids"].tolist()
            if int(t) not in special_ids
        ]
        if toks:
            btoks.append(toks)

    num_constraints = len(ctoks)
    total_constraint_tokens = sum(len(t) for t in ctoks)

    src_len = int(attention_mask.sum().item())
    max_steps = min(max_length, 2 * src_len + total_constraint_tokens + 16)

    # the encoder runs exactly once; its states are re-used every step
    with torch.no_grad():
        enc = model.get_encoder()(input_ids=input_ids, attention_mask=attention_mask)

    def coverage(h):
        cov = sum(len(ctoks[j]) for j in h["done"])
        if h["open"] is not None:
            cov += h["open"][1]
        return cov

    def violates_blocklist(ids, tok):
        # negative: would appending `tok` complete a banned sequence?
        candidate = ids + [tok]
        for banned in btoks:
            if len(candidate) >= len(banned) and candidate[-len(banned):] == banned:
                return True
        return False

    def blocked_next_tokens(ids):
        # negative: the tokens that would complete a banned sequence at
        # this step, given the hypothesis's current tail
        blocked = set()
        for banned in btoks:
            prefix = banned[:-1]
            if not prefix or (len(ids) >= len(prefix) and ids[-len(prefix):] == prefix):
                blocked.add(banned[-1])
        return blocked

    init = {
        "ids": [start_id],
        "score": 0.0,
        "done": frozenset(),
        "open": None,
    }

    grid = {0: [init]}
    completed = []

    for step in range(max_steps):
        remaining = max_steps - step

        # prune hypotheses that can no longer cover all constraints in time
        active = [
            h
            for hs in grid.values()
            for h in hs
            if (total_constraint_tokens - coverage(h)) <= remaining
        ]

        if not active:
            break

        dec = torch.tensor(
            [h["ids"] for h in active],
            dtype=torch.long,
            device=device,
        )

        exp_enc = BaseModelOutput(
            last_hidden_state=enc.last_hidden_state.expand(len(active), -1, -1)
        )

        with torch.no_grad():
            dec_out = model.model(
                encoder_outputs=exp_enc,
                attention_mask=attention_mask.expand(len(active), -1),
                decoder_input_ids=dec,
            )

            last_hidden = dec_out.last_hidden_state[:, -1, :]
            logits = model.lm_head(last_hidden) + model.final_logits_bias

        logp = F.log_softmax(logits, dim=-1)
        new_grid = {}

        def add(h):
            new_grid.setdefault(coverage(h), []).append(h)

        for idx, h in enumerate(active):
            c = coverage(h)
            lp = logp[idx]
            must_cover = (total_constraint_tokens - c) >= remaining

            # CONTINUE: an open constraint must be finished before anything else
            if h["open"] is not None:
                j, p = h["open"]
                tok = ctoks[j][p]

                # negative: drop the expansion if the forced token would
                # complete a banned sequence
                if violates_blocklist(h["ids"], tok):
                    continue

                nxt = p + 1
                fin = h["done"] | {j} if nxt == len(ctoks[j]) else h["done"]

                add(
                    {
                        "ids": h["ids"] + [tok],
                        "score": h["score"] + lp[tok].item(),
                        "done": fin,
                        "open": None if nxt == len(ctoks[j]) else (j, nxt),
                    }
                )
                continue

            # GENERATE: free tokens, only while there is still time to cover the rest
            if not must_cover:
                gen_lp = lp.clone()
                gen_lp[pad_id] = float("-inf")

                if c < total_constraint_tokens:
                    gen_lp[eos_id] = float("-inf")

                # negative: mask any token that would complete a banned
                # sequence, so topk can only promote allowed continuations
                for tok in blocked_next_tokens(h["ids"]):
                    gen_lp[tok] = float("-inf")

                top_lp, top_ids = torch.topk(gen_lp, num_beams)

                for k in range(num_beams):
                    s = top_lp[k].item()

                    if s == float("-inf"):
                        continue

                    tok = int(top_ids[k])

                    nh = {
                        "ids": h["ids"] + [tok],
                        "score": h["score"] + s,
                        "done": h["done"],
                        "open": None,
                    }

                    if tok == eos_id:
                        completed.append(nh)
                    else:
                        add(nh)

            # START: open any not-yet-covered constraint
            for j in range(num_constraints):
                if j in h["done"]:
                    continue

                tok = ctoks[j][0]

                # negative: same guard for a constraint's first token
                if violates_blocklist(h["ids"], tok):
                    continue

                nxt = 1
                fin = h["done"] | {j} if nxt == len(ctoks[j]) else h["done"]

                add(
                    {
                        "ids": h["ids"] + [tok],
                        "score": h["score"] + lp[tok].item(),
                        "done": fin,
                        "open": None if nxt == len(ctoks[j]) else (j, nxt),
                    }
                )

        grid = {}

        for c, hs in new_grid.items():
            hs.sort(key=lambda x: x["score"], reverse=True)
            # keep the top-k *distinct* hypotheses per coverage level
            seen, uniq = set(), []
            for h in hs:
                key = (tuple(h["ids"]), h["open"], h["done"])
                if key in seen:
                    continue
                seen.add(key)
                uniq.append(h)
                if len(uniq) == num_beams:
                    break
            grid[c] = uniq

    def norm(h):
        length = len(h["ids"]) - 1
        return h["score"] / (length ** length_penalty) if length > 0 else h["score"]

    if completed:
        pool = completed
    elif total_constraint_tokens in grid and grid[total_constraint_tokens]:
        pool = grid[total_constraint_tokens]
    else:
        pool = grid[max(grid)]

    best = max(pool, key=norm)
    return torch.tensor([best["ids"]], dtype=torch.long, device=device)

In [51]:
## Track B: pick-revise with Grid Beam Search
# GBS batches the whole (timestep x coverage) grid through the model at every
# step, so this is the slow cell -- lower N_EVAL above if you need a quick pass.
gbs_outputs = run_pick_revise(grid_beam_search, "GRID BEAM SEARCH (Hokamp & Liu 2017)", base_outputs)


GRID BEAM SEARCH (Hokamp & Liu 2017)
base    BLEU= 25.45


GRID BEAM SEARCH (Hokamp & Liu 2017) cycle 1: 100%|██████████| 100/100 [03:10<00:00,  1.90s/it]


cycle 1  BLEU= 31.81   constraints= 99   satisfied(tok)=100.0%   in_text=100.0%   mean_len= 31.7
    [0] constraint='Премьер-министры Индии и'  tok=True  text=True
        before: Встреча премьер-министров Индии и Японии в Токио
        after : Премьер-министры Индии и Японии встречаются в Токио
    [1] constraint='Новый премьер-министр Индии,'  tok=True  text=True
        before: Новый премьер-министр Индии Нарендра Моди встречается со своим японским коллегой, Синзо Абэ, в Токио, чтобы обсудить экономические связи и связи в области безопасности в ходе своего первого крупного визита за рубеж после победы на выборах в мае.
        after : Новый премьер-министр Индии, Нарендра Моди, встречается со своим японским коллегой, Синзо Абэ, в Токио, чтобы обсудить экономические связи и связи в области безопасности в ходе своего первого крупного визита за рубеж после победы на выборах в мае.
    [2] constraint='Господин Моди находится'  tok=True  text=True
        before: Мистер Моди в пятидневно

GRID BEAM SEARCH (Hokamp & Liu 2017) cycle 2: 100%|██████████| 100/100 [05:33<00:00,  3.33s/it]


cycle 2  BLEU= 38.41   constraints= 94   satisfied(tok)=100.0%   in_text=100.0%   mean_len= 37.4


GRID BEAM SEARCH (Hokamp & Liu 2017) cycle 3: 100%|██████████| 100/100 [07:27<00:00,  4.47s/it]


cycle 3  BLEU= 40.40   constraints= 77   satisfied(tok)=100.0%   in_text=100.0%   mean_len= 43.2


GRID BEAM SEARCH (Hokamp & Liu 2017) cycle 4: 100%|██████████| 100/100 [08:40<00:00,  5.20s/it]

cycle 4  BLEU= 40.59   constraints= 63   satisfied(tok)=100.0%   in_text=100.0%   mean_len= 48.8


## Section 8 — Fulfilling Negative Constraints: Grid Beam Search

Sections 5–7 answered one question: how do we force a phrase *in*? The mirror question arrives immediately in any real use: how do we keep a phrase *out*? Profanity that should give way to a euphemism, a competitor's name, an anglicism the style guide forbids. The machinery is already on the table — one more masking rule finishes the job.

This section steps away from pick–revise, deliberately. Pick–revise exists to *recover reference content*: it mines phrases the reference has and the hypothesis lacks. A ban has no analogue in that protocol — the reference contains nothing we want excluded, and banning reference phrases would push BLEU down by construction, measuring nothing. So negative constraints are demonstrated here the way they would actually be used: bare phrases a user wants excluded, no reference in sight. (`DEMO.ipynb` wraps exactly this interface for interactive use.)

The mechanism, traced through the Section 7 decoder: a banned phrase goes through `encode_target()` into token-id sequences (`btoks`) → at every **generate** expansion, any token that would *complete* a banned sequence — given the hypothesis's current tail — has its log-probability masked to `-inf` before the top-k (`blocked_next_tokens`) → the probability mass the model gave the banned continuation falls to its runner-up, and because the top of the distribution at that point is dominated by alternative wordings of the same source meaning, the runner-up for a harsh word is usually a milder synonym → forced tokens are guarded too (`violates_blocklist`): a **start**/**continue** expansion that would complete a ban is dropped instead of emitted.

Design notes:

- **No new grid dimension.** Positive constraints needed coverage levels to *protect* improbable-but-required progress from beam pruning. A ban protects nothing — it only removes options — so it lives entirely inside the per-step masking, at negligible cost.
- **The match is exact at the token level.** Banning «Мистер» bans that segmentation only, not «мистера» or any other case form. A production blocklist expands each banned lemma into its surface variants (or adds a substring check on the detokenized output as a safety net).
- **Banning selects *against*, never *for*.** If the model's runner-up is another word you dislike, that is what you get. The fix is composition: ban the unwanted phrase *and* require the replacement — both constraint types in one call, shown below.
- **Don't require what you ban.** A positive constraint containing a banned sequence makes the two unsatisfiable together; the search can only return a fallback hypothesis.

The demonstration reuses sentence `[2]` of the evaluation subset, familiar from the cycle-1 examples above. Unconstrained decoding renders "Mr Modi" with the anglicism «Мистер»; banning that one word forces the decoder onto its own next-best honorific, and adding a positive constraint on top directs the choice instead of trusting the model's ranking.

In [ ]:
## Negative constraints, standalone: ban a phrase and the runner-up takes its place
def phrases_to_constraints(phrases):
    """Wrap target-language phrases in the {token_ids} dicts the decoders expect."""
    return [
        {
            "token_ids": encode_target(
                p, add_special_tokens=False, return_tensors="pt"
            )["input_ids"][0]
        }
        for p in phrases
    ]


demo_idx = 2  # "Mr Modi is on a five-day trip..." -- decoded unconstrained, "Mr" comes out as «Мистер»
demo_src = src_ids[demo_idx:demo_idx + 1]
demo_mask = src_mask[demo_idx:demo_idx + 1]

banned = ["Мистер"]           # the anglicism to keep out
required = ["Господин Моди"]  # optional: direct the replacement instead of trusting the runner-up

print("source  :", src_texts[demo_idx])

free_ids = grid_beam_search(demo_src, demo_mask, [])
print("free    :", tokenizer.batch_decode(free_ids, skip_special_tokens=True)[0])

neg_ids = grid_beam_search(
    demo_src, demo_mask, [],
    negative_constraints=phrases_to_constraints(banned),
)
print("banned  :", tokenizer.batch_decode(neg_ids, skip_special_tokens=True)[0])

both_ids = grid_beam_search(
    demo_src, demo_mask,
    phrases_to_constraints(required),
    negative_constraints=phrases_to_constraints(banned),
)
print("directed:", tokenizer.batch_decode(both_ids, skip_special_tokens=True)[0])

# The exclusion is checkable, not just visible
for p in banned:
    b_ids = strip_special(encode_target(p, add_special_tokens=False)["input_ids"])
    assert not is_contiguous_subseq(b_ids, strip_special(neg_ids[0].tolist()))
    assert not is_contiguous_subseq(b_ids, strip_special(both_ids[0].tolist()))
print("check   : no banned token sequence appears in the constrained outputs")